# In-Class Coding — VAE & GAN
### Lec7 · Spring 2026

**Instructions**
- Work through each problem in order. Later problems build on earlier ones.
- Every cell marked `# ── YOUR CODE HERE ──` requires you to fill in the missing logic.
- Do **not** use pre-built VAE/GAN libraries — implement the key components from scratch.
- ⭐ problems are advanced / bonus; attempt them after finishing the main problems.
- Run the **test cells** after each implementation to check your answer.

---
**This notebook has two sections:**
1. Variational Autoencoder (Problems 1–5)
2. Generative Adversarial Network (Problems 6–9)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_moons

torch.manual_seed(42)
np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print('Ready.')

---
# Section 1 — Variational Autoencoder

## Problem 1 — The Reparameterisation Trick

The VAE encoder outputs two vectors $\boldsymbol{\mu}$ and $\log\boldsymbol{\sigma}^2$ (both shape $B \times d$).
To sample a latent code we use:

$$z = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\varepsilon}, \qquad \boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0}, I)$$

where $\boldsymbol{\sigma} = \exp\!\left(\tfrac{1}{2}\log\boldsymbol{\sigma}^2\right)$.

This keeps the **sampling step outside the gradient path** (the gradient flows through $\boldsymbol{\mu}$ and $\log\boldsymbol{\sigma}^2$, not through the random $\boldsymbol{\varepsilon}$).

Implement `reparameterise(mu, log_var)` — do **not** call `torch.distributions`.

In [ ]:
def reparameterise(mu, log_var):
    """
    Draw z ~ N(mu, sigma^2) using the reparameterisation trick.

    Parameters
    ----------
    mu      : torch.Tensor, shape (B, d)  — posterior mean
    log_var : torch.Tensor, shape (B, d)  — log of posterior variance

    Returns
    -------
    z : torch.Tensor, shape (B, d)  — sampled latent code
    """
    # ── YOUR CODE HERE ──────────────────────────────────────────────────────
    # Step 1: compute sigma from log_var
    # Step 2: sample eps ~ N(0, I) with the same shape as mu
    # Step 3: return mu + sigma * eps


    # ────────────────────────────────────────────────────────────────────────

In [ ]:
# ── Test P1 ───────────────────────────────────────────────────────────────────
torch.manual_seed(0)
B, d = 50_000, 4
mu_t      = torch.tensor([[1.0, -2.0, 0.5, 3.0]]).expand(B, -1)
log_var_t = torch.tensor([[0.0,  0.0, 0.0, 0.0]]).expand(B, -1)  # sigma=1

z_t = reparameterise(mu_t, log_var_t)

assert z_t.shape == (B, d), f'P1 FAIL: shape should be ({B},{d}), got {z_t.shape}'
assert torch.allclose(z_t.mean(0), mu_t[0], atol=0.05), \
    f'P1 FAIL: empirical mean {z_t.mean(0).tolist()} far from mu {mu_t[0].tolist()}'
assert torch.allclose(z_t.std(0),  torch.ones(d), atol=0.05), \
    f'P1 FAIL: empirical std {z_t.std(0).tolist()} should be ~1 when log_var=0'

# Gradient must flow through mu
mu_g      = torch.zeros(2, 3, requires_grad=True)
log_var_g = torch.zeros(2, 3)
z_g = reparameterise(mu_g, log_var_g)
z_g.sum().backward()
assert mu_g.grad is not None, 'P1 FAIL: gradient did not flow through mu'

print('P1 PASSED ✓')

# Visualise: 2D scatter of samples vs. true distribution
torch.manual_seed(1)
mu2      = torch.tensor([[2.0, -1.0]]).expand(2000, -1)
log_var2 = torch.log(torch.tensor([[0.25, 1.0]])).expand(2000, -1)  # sigma=[0.5, 1.0]
z2 = reparameterise(mu2, log_var2).detach().numpy()

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(z2[:, 0], z2[:, 1], s=5, alpha=0.3, label='z samples')
ax.scatter([2.0], [-1.0], s=100, c='red', zorder=5, label='μ = (2, -1)')
ax.set_aspect('equal')
ax.set_title('P1 — Reparameterised samples  (σ₁=0.5, σ₂=1.0)')
ax.legend(); plt.tight_layout(); plt.show()

## Problem 2 — KL Divergence (Gaussian Posterior vs. Standard Normal Prior)

The KL regularisation term in the ELBO is:

$$D_{\text{KL}}\!\left(q_\phi(z|x)\,\|\,p(z)\right) = D_{\text{KL}}\!\left(\mathcal{N}(\boldsymbol{\mu}, \text{diag}(\boldsymbol{\sigma}^2))\,\|\,\mathcal{N}(\mathbf{0}, I)\right)$$

For diagonal Gaussians this has a **closed form** (derivable by direct integration):

$$D_{\text{KL}} = \frac{1}{2}\sum_{j=1}^{d}\left(\mu_j^2 + \sigma_j^2 - \log\sigma_j^2 - 1\right)$$

Implement `kl_divergence(mu, log_var)`. Return a **scalar** (summed over $d$, **averaged** over the batch).

In [ ]:
def kl_divergence(mu, log_var):
    """
    Closed-form KL divergence KL( N(mu, diag(exp(log_var))) || N(0, I) ).

    Parameters
    ----------
    mu      : torch.Tensor, shape (B, d)
    log_var : torch.Tensor, shape (B, d)

    Returns
    -------
    scalar : KL averaged over the batch (summed over d)
    """
    # ── YOUR CODE HERE ──────────────────────────────────────────────────────
    # Hint: sigma^2 = exp(log_var), so log(sigma^2) = log_var
    # Per-dimension contribution: 0.5 * (mu^2 + exp(log_var) - log_var - 1)
    # Sum over d, mean over B.


    # ────────────────────────────────────────────────────────────────────────

In [ ]:
# ── Test P2 ───────────────────────────────────────────────────────────────────

# When mu=0 and log_var=0 (i.e. posterior = prior), KL must be 0
mu0   = torch.zeros(10, 4)
lv0   = torch.zeros(10, 4)
kl0   = kl_divergence(mu0, lv0)
assert abs(kl0.item()) < 1e-6, f'P2 FAIL: KL(N(0,I)||N(0,I)) should be 0, got {kl0.item():.6f}'

# KL must always be non-negative
torch.manual_seed(5)
mu_rand   = torch.randn(32, 8)
lv_rand   = torch.randn(32, 8)
kl_rand   = kl_divergence(mu_rand, lv_rand)
assert kl_rand.item() >= 0, f'P2 FAIL: KL must be >= 0, got {kl_rand.item():.4f}'

# Analytical check: KL(N(1,1)||N(0,1)) = 0.5*(1+1-0-1) = 0.5 per dim
mu1   = torch.ones(1, 1)       # mu = 1
lv1   = torch.zeros(1, 1)      # log_var = 0  =>  sigma^2 = 1
kl1   = kl_divergence(mu1, lv1)
assert abs(kl1.item() - 0.5) < 1e-5, \
    f'P2 FAIL: KL(N(1,1)||N(0,1)) should be 0.5, got {kl1.item():.6f}'

# Monte Carlo verification (large sample)
torch.manual_seed(99)
mu_mc   = torch.tensor([[2.0, -1.0]])
lv_mc   = torch.tensor([[0.0,  0.5]])   # sigma = [1.0, exp(0.25)]
kl_ana  = kl_divergence(mu_mc, lv_mc).item()
z_mc    = reparameterise(mu_mc.expand(200_000, -1), lv_mc.expand(200_000, -1))
# Monte Carlo: E_q[log q - log p]
log_q   = torch.distributions.Normal(mu_mc, (0.5*lv_mc).exp()).log_prob(z_mc).sum(-1)
log_p   = torch.distributions.Normal(0, 1).log_prob(z_mc).sum(-1)
kl_mc   = (log_q - log_p).mean().item()
assert abs(kl_ana - kl_mc) < 0.02, \
    f'P2 FAIL: analytical KL={kl_ana:.4f} vs MC KL={kl_mc:.4f}'

print(f'P2 PASSED ✓  (analytical={kl_ana:.4f}, MC≈{kl_mc:.4f})')

## Problem 3 — ELBO Loss

The VAE training objective is the **Evidence Lower BOund (ELBO)**:

$$\mathcal{L}_{\text{ELBO}} = \underbrace{-\mathbb{E}_{z \sim q_\phi}[\log p_\theta(x|z)]}_{\text{reconstruction loss}} + \beta \cdot \underbrace{D_{\text{KL}}(q_\phi(z|x)\|p(z))}_{\text{KL regularisation}}$$

- **Reconstruction loss:** binary cross-entropy (BCE) averaged over pixels, then over the batch:
$$\mathcal{L}_{\text{recon}} = -\frac{1}{B}\sum_{i=1}^B \sum_{j=1}^D \left[x_{ij}\log\hat{x}_{ij} + (1-x_{ij})\log(1-\hat{x}_{ij})\right]$$
- **KL loss:** your `kl_divergence` from Problem 2.
- $\beta \geq 0$ controls regularisation strength ($\beta=1$ is the standard VAE).

Implement `elbo_loss(x, x_hat, mu, log_var, beta=1.0)`.

In [ ]:
def elbo_loss(x, x_hat, mu, log_var, beta=1.0):
    """
    ELBO loss = reconstruction_loss + beta * kl_loss.

    Parameters
    ----------
    x       : torch.Tensor, shape (B, D)  — original input, values in [0,1]
    x_hat   : torch.Tensor, shape (B, D)  — reconstruction (sigmoid output), values in [0,1]
    mu      : torch.Tensor, shape (B, d)
    log_var : torch.Tensor, shape (B, d)
    beta    : float  — KL weight (default 1.0)

    Returns
    -------
    loss        : scalar — total ELBO loss
    recon_loss  : scalar — reconstruction term only
    kl          : scalar — KL term only
    """
    # ── YOUR CODE HERE ──────────────────────────────────────────────────────
    # Hint: use nn.functional.binary_cross_entropy (NOT binary_cross_entropy_with_logits)
    #       with reduction='sum', then divide by B to average over the batch.


    # ────────────────────────────────────────────────────────────────────────

In [ ]:
# ── Test P3 ───────────────────────────────────────────────────────────────────
import torch.nn.functional as F

torch.manual_seed(3)
B, D, d = 16, 20, 4
x_t     = torch.rand(B, D).round()          # binary
x_hat_t = torch.sigmoid(torch.randn(B, D))  # in (0,1)
mu_t    = torch.randn(B, d)
lv_t    = torch.randn(B, d)

loss, recon, kl = elbo_loss(x_t, x_hat_t, mu_t, lv_t, beta=1.0)

# Shape checks
assert loss.shape == torch.Size([]),  'P3 FAIL: loss must be a scalar'
assert recon.shape == torch.Size([]), 'P3 FAIL: recon must be a scalar'
assert kl.shape == torch.Size([]),    'P3 FAIL: kl must be a scalar'

# KL must be non-negative
assert kl.item() >= 0, f'P3 FAIL: KL should be >= 0, got {kl.item():.4f}'

# With beta=0, total loss should equal recon loss
loss0, recon0, _ = elbo_loss(x_t, x_hat_t, mu_t, lv_t, beta=0.0)
assert abs(loss0.item() - recon0.item()) < 1e-5, \
    'P3 FAIL: with beta=0, total loss must equal recon loss'

# Recon loss should match nn.BCELoss reference
ref_recon = F.binary_cross_entropy(x_hat_t, x_t, reduction='sum') / B
assert abs(recon.item() - ref_recon.item()) < 1e-4, \
    f'P3 FAIL: recon={recon.item():.4f} vs reference={ref_recon.item():.4f}'

# Total loss = recon + beta * kl
beta_test = 2.0
loss2, recon2, kl2 = elbo_loss(x_t, x_hat_t, mu_t, lv_t, beta=beta_test)
assert abs(loss2.item() - (recon2.item() + beta_test * kl2.item())) < 1e-4, \
    'P3 FAIL: loss should equal recon + beta*kl'

print(f'P3 PASSED ✓  (loss={loss.item():.2f}, recon={recon.item():.2f}, kl={kl.item():.2f})')

## Problem 4 — Train a VAE on 2D Moons

We fit a VAE to the **two-moons** dataset (2D, two crescent-shaped clusters).
The dataset is already binarised so BCE reconstruction loss applies directly.

Your tasks:
1. Complete the `forward()` method of `VAE2D` — it should call the encoder, reparameterise, and decode.
2. Complete the `train_step()` function — compute the ELBO loss and take one gradient step.
3. Run the training loop and observe the latent space.

**Architecture:**
- Encoder: Linear(2→32) → ReLU → Linear(32→32) → ReLU → two heads: Linear(32→`latent_dim`) for μ and log σ²
- Decoder: Linear(`latent_dim`→32) → ReLU → Linear(32→32) → ReLU → Linear(32→2) → Sigmoid

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
np.random.seed(0)
X_moons, y_moons = make_moons(n_samples=2000, noise=0.05)

# Normalise to [0,1] for BCE
X_min, X_max  = X_moons.min(0), X_moons.max(0)
X_norm        = (X_moons - X_min) / (X_max - X_min + 1e-8)

X_t = torch.tensor(X_norm, dtype=torch.float32)
y_t = torch.tensor(y_moons)

moon_dl = DataLoader(TensorDataset(X_t, y_t), batch_size=128, shuffle=True)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X_norm[y_moons==0, 0], X_norm[y_moons==0, 1], s=8, label='Moon 0')
ax.scatter(X_norm[y_moons==1, 0], X_norm[y_moons==1, 1], s=8, label='Moon 1')
ax.set_title('Two-moons dataset (normalised)'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
LATENT_DIM = 2

class VAE2D(nn.Module):
    def __init__(self, input_dim=2, latent_dim=LATENT_DIM, hidden=32):
        super().__init__()
        self.shared_enc = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.ReLU(),
        )
        self.fc_mu      = nn.Linear(hidden, latent_dim)
        self.fc_log_var = nn.Linear(hidden, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),     nn.ReLU(),
            nn.Linear(hidden, input_dim),  nn.Sigmoid(),
        )

    def encode(self, x):
        h       = self.shared_enc(x)
        mu      = self.fc_mu(h)
        log_var = self.fc_log_var(h)
        return mu, log_var

    def forward(self, x):
        """
        Returns
        -------
        x_hat   : reconstructed input, shape (B, 2)
        mu      : posterior mean,       shape (B, latent_dim)
        log_var : posterior log-var,    shape (B, latent_dim)
        """
        # ── YOUR CODE HERE ──────────────────────────────────────────────────
        # 1. Call self.encode(x) to get mu and log_var
        # 2. Call reparameterise(mu, log_var) to get z
        # 3. Call self.decoder(z) to get x_hat
        # 4. Return x_hat, mu, log_var


        # ────────────────────────────────────────────────────────────────────


def train_step(model, optimiser, x, beta=1.0):
    """
    One gradient update on a batch x.
    Returns the scalar total loss.
    """
    # ── YOUR CODE HERE ──────────────────────────────────────────────────────
    # 1. Forward pass
    # 2. Compute elbo_loss
    # 3. Zero gradients, backprop, step
    # 4. Return loss.item()


    # ────────────────────────────────────────────────────────────────────────


vae = VAE2D(latent_dim=LATENT_DIM).to(device)
print('VAE params:', sum(p.numel() for p in vae.parameters()))

In [ ]:
# ── Training loop (run this cell) ─────────────────────────────────────────────
optimiser  = optim.Adam(vae.parameters(), lr=1e-3)
N_EPOCHS   = 200
loss_hist  = []

for epoch in range(1, N_EPOCHS + 1):
    epoch_loss = 0.0
    for xb, _ in moon_dl:
        xb = xb.to(device)
        epoch_loss += train_step(vae, optimiser, xb, beta=1.0)
    loss_hist.append(epoch_loss / len(moon_dl))
    if epoch % 50 == 0:
        print(f'Epoch {epoch:3d}/{N_EPOCHS}  loss={loss_hist[-1]:.4f}')

print('Training complete.')

In [ ]:
# ── Test P4 ───────────────────────────────────────────────────────────────────
# 1. Loss should be finite and generally decreasing
assert all(np.isfinite(loss_hist)), 'P4 FAIL: loss contains NaN or Inf'
assert loss_hist[-1] < loss_hist[0], \
    f'P4 FAIL: loss did not decrease: start={loss_hist[0]:.3f}, end={loss_hist[-1]:.3f}'

# 2. forward() must return three tensors of the right shape
vae.eval()
with torch.no_grad():
    x_hat_chk, mu_chk, lv_chk = vae(X_t[:8].to(device))
assert x_hat_chk.shape == (8, 2), f'P4 FAIL: x_hat shape {x_hat_chk.shape}'
assert mu_chk.shape    == (8, LATENT_DIM), f'P4 FAIL: mu shape {mu_chk.shape}'

print('P4 PASSED ✓')

# ── Visualise latent space ────────────────────────────────────────────────────
vae.eval()
with torch.no_grad():
    mu_all, lv_all = vae.encode(X_t.to(device))
    z_all = reparameterise(mu_all, lv_all).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(loss_hist)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('ELBO loss')
axes[0].set_title('Training loss')

sc = axes[1].scatter(z_all[:, 0], z_all[:, 1], c=y_moons, s=10,
                     cmap='coolwarm', alpha=0.6)
plt.colorbar(sc, ax=axes[1], label='Moon label')
axes[1].set_title(f'VAE latent space (latent_dim={LATENT_DIM})')

plt.tight_layout(); plt.show()
print('Observation: the two moons should appear as two separated clusters in latent space.')

## Problem 5 — Latent Space Interpolation

One hallmark of a well-trained VAE: the **latent space is smooth and continuous**.
Any straight line between two encoded points should decode to a meaningful trajectory.

**Task:** Choose one point from each moon. Encode both to get $z_A$ and $z_B$ (use the mean $\boldsymbol{\mu}$, not a sample).
Interpolate $z_t = (1-t)\,z_A + t\,z_B$ for $t \in \{0, 0.1, \ldots, 1.0\}$ (11 steps).
Decode each $z_t$ and plot the resulting 2D points as a trajectory over the data.

**Requirement:** `interpolate(vae, x_a, x_b, steps)` must return a tensor of shape `(steps, 2)`.

In [ ]:
def interpolate(model, x_a, x_b, steps=11):
    """
    Linearly interpolate in latent space between encoded x_a and x_b.

    Parameters
    ----------
    model  : trained VAE2D
    x_a    : torch.Tensor, shape (1, 2) — start point
    x_b    : torch.Tensor, shape (1, 2) — end point
    steps  : int — number of interpolation steps (including endpoints)

    Returns
    -------
    decoded : torch.Tensor, shape (steps, 2) — decoded points along the path
    """
    model.eval()
    with torch.no_grad():
        # ── YOUR CODE HERE ──────────────────────────────────────────────────
        # 1. Encode x_a → mu_a (use the mean, not a sample)
        # 2. Encode x_b → mu_b
        # 3. For each t in linspace(0,1,steps): z_t = (1-t)*mu_a + t*mu_b
        # 4. Decode all z_t and return as a (steps, 2) tensor


        # ────────────────────────────────────────────────────────────────────

In [ ]:
# ── Test P5 ───────────────────────────────────────────────────────────────────
# Pick one point from each moon
idx_a = np.where(y_moons == 0)[0][0]
idx_b = np.where(y_moons == 1)[0][0]
x_a   = X_t[idx_a:idx_a+1].to(device)
x_b   = X_t[idx_b:idx_b+1].to(device)

path = interpolate(vae, x_a, x_b, steps=11)

assert path.shape == (11, 2), f'P5 FAIL: shape should be (11,2), got {path.shape}'
assert torch.allclose(path[0],  vae.encode(x_a)[0][0].detach(), atol=1e-4), \
    'P5 FAIL: first decoded point should match decoding of mu_a'

print('P5 PASSED ✓')

# ── Plot trajectory ───────────────────────────────────────────────────────────
path_np = path.cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: trajectory in input space
axes[0].scatter(X_norm[y_moons==0,0], X_norm[y_moons==0,1], s=6, alpha=0.3, c='steelblue')
axes[0].scatter(X_norm[y_moons==1,0], X_norm[y_moons==1,1], s=6, alpha=0.3, c='tomato')
axes[0].plot(path_np[:, 0], path_np[:, 1], 'ko-', ms=6, lw=1.5, label='Interpolation path')
axes[0].scatter(*x_a.cpu().numpy()[0], s=100, c='blue',  zorder=5, label='x_a')
axes[0].scatter(*x_b.cpu().numpy()[0], s=100, c='red',   zorder=5, label='x_b')
axes[0].set_title('Decoded interpolation in input space'); axes[0].legend(fontsize=8)

# Right: trajectory in latent space
axes[1].scatter(z_all[y_moons==0, 0], z_all[y_moons==0, 1], s=6, alpha=0.3, c='steelblue')
axes[1].scatter(z_all[y_moons==1, 0], z_all[y_moons==1, 1], s=6, alpha=0.3, c='tomato')
with torch.no_grad():
    mu_a = vae.encode(x_a)[0].cpu().numpy()[0]
    mu_b = vae.encode(x_b)[0].cpu().numpy()[0]
ts = np.linspace(0, 1, 11)
lat_path = np.outer(1 - ts, mu_a) + np.outer(ts, mu_b)
axes[1].plot(lat_path[:, 0], lat_path[:, 1], 'ko-', ms=6, lw=1.5)
axes[1].scatter(*mu_a, s=100, c='blue', zorder=5)
axes[1].scatter(*mu_b, s=100, c='red',  zorder=5)
axes[1].set_title('Linear path in latent space')

plt.tight_layout(); plt.show()

# ⭐ Advanced: generate new samples from the prior
print('\n⭐ Advanced: sample from the prior p(z) = N(0,I) and decode.')
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────
# 1. Sample 200 points z_prior ~ N(0, I)  (shape: (200, LATENT_DIM))
# 2. Decode with vae.decoder
# 3. Scatter-plot decoded points over the real data


# ─────────────────────────────────────────────────────────────────────────────

---
# Section 2 — Generative Adversarial Network

In [ ]:
# ── Shared data for Section 2: 8-mode ring dataset ────────────────────────────
def make_ring(n=2000, radius=2.0, n_modes=8, sigma=0.05, seed=0):
    np.random.seed(seed)
    angles  = np.linspace(0, 2*np.pi, n_modes, endpoint=False)
    centers = np.stack([radius*np.cos(angles), radius*np.sin(angles)], axis=1)
    z       = np.random.choice(n_modes, size=n)
    data    = centers[z] + sigma * np.random.randn(n, 2)
    return data.astype(np.float32), centers

ring_data, ring_centers = make_ring()

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(ring_data[:, 0], ring_data[:, 1], s=5, alpha=0.4, c='steelblue')
ax.scatter(ring_centers[:, 0], ring_centers[:, 1], s=60, c='red', zorder=5, label='Mode centers')
ax.set_aspect('equal'); ax.set_title('Ring dataset — 8 modes'); ax.legend()
plt.tight_layout(); plt.show()

## Problem 6 — GAN Loss Functions

Recall the minimax objective (non-saturating form):

$$\mathcal{L}_D = -\mathbb{E}_{x \sim p_{\text{data}}}[\log D(x)] - \mathbb{E}_{z}[\log(1 - D(G(z)))]$$

$$\mathcal{L}_G = -\mathbb{E}_{z}[\log D(G(z))]$$

Both are **binary cross-entropy** problems:
- $D$ labels real images as 1 and fake images as 0
- $G$ wants $D$ to label its fakes as 1

Implement `discriminator_loss(d_real, d_fake)` and `generator_loss(d_fake)` using **only** `torch.log` — do not call `nn.BCELoss`.
Both inputs are already sigmoid outputs $\in (0, 1)$. Return scalars (batch means).

In [ ]:
def discriminator_loss(d_real, d_fake):
    """
    D loss = -E[log D(real)] - E[log(1 - D(fake))]

    Parameters
    ----------
    d_real : torch.Tensor, shape (B,) or (B,1)  — D output on real samples
    d_fake : torch.Tensor, shape (B,) or (B,1)  — D output on fake samples

    Returns
    -------
    scalar loss
    """
    # ── YOUR CODE HERE ──────────────────────────────────────────────────────


    # ────────────────────────────────────────────────────────────────────────


def generator_loss(d_fake):
    """
    G loss (non-saturating) = -E[log D(fake)]

    Parameters
    ----------
    d_fake : torch.Tensor, shape (B,) or (B,1)  — D output on fake samples

    Returns
    -------
    scalar loss
    """
    # ── YOUR CODE HERE ──────────────────────────────────────────────────────


    # ────────────────────────────────────────────────────────────────────────

In [ ]:
# ── Test P6 ───────────────────────────────────────────────────────────────────

# Scenario 1: perfect D — D(real)=1, D(fake)=0 → D loss should be ~0
eps = 1e-7
d_real_perf = torch.full((16,), 1 - eps)
d_fake_perf = torch.full((16,), eps)
loss_D_perf = discriminator_loss(d_real_perf, d_fake_perf)
assert loss_D_perf.item() < 0.01, \
    f'P6 FAIL: perfect D should have ~0 loss, got {loss_D_perf.item():.4f}'

# Scenario 2: confused D — D(real)=D(fake)=0.5 → D loss should be ~log(2) ≈ 0.693
d_half = torch.full((16,), 0.5)
loss_D_half = discriminator_loss(d_half, d_half)
assert abs(loss_D_half.item() - np.log(2)) < 0.01, \
    f'P6 FAIL: D at chance → loss should be log(2)≈0.693, got {loss_D_half.item():.4f}'

# Scenario 3: G loss — D(fake)=1 (G fools D) → G loss should be ~0
loss_G_fool = generator_loss(d_real_perf)   # D says fake is real
assert loss_G_fool.item() < 0.01, \
    f'P6 FAIL: G perfectly fools D → G loss should be ~0, got {loss_G_fool.item():.4f}'

# Scenario 4: G loss — D(fake)=0 (D perfectly rejects) → G loss large
loss_G_fail = generator_loss(d_fake_perf)
assert loss_G_fail.item() > 5.0, \
    f'P6 FAIL: D perfectly rejects G → G loss should be large, got {loss_G_fail.item():.4f}'

# At equilibrium D(x)=0.5 for all x: both losses should equal log(2)
loss_G_half = generator_loss(d_half)
assert abs(loss_G_half.item() - np.log(2)) < 0.01, \
    f'P6 FAIL: G at equilibrium → loss should be log(2), got {loss_G_half.item():.4f}'

print(f'P6 PASSED ✓')
print(f'  D@perfect={loss_D_perf.item():.4f}  D@half={loss_D_half.item():.4f}  '
      f'G@fools={loss_G_fool.item():.4f}  G@half={loss_G_half.item():.4f}')
print(f'  log(2) = {np.log(2):.4f}  (equilibrium value)')

## Problem 7 — Train a GAN on the Ring Dataset

Using your loss functions from Problem 6, complete the GAN training loop.

The standard GAN training alternates:
1. **Update D** (freeze G): sample real batch + fake batch, compute $\mathcal{L}_D$, step.
2. **Update G** (freeze D): sample new fake batch, compute $\mathcal{L}_G$ using current D, step.

**Key detail:** use `.detach()` on fake images when training D — otherwise gradients flow into G prematurely.

Complete `gan_train_step(G, D, opt_G, opt_D, real_batch)`.

In [ ]:
LATENT_DIM_GAN = 2

def make_G(latent_dim=LATENT_DIM_GAN, width=64):
    return nn.Sequential(
        nn.Linear(latent_dim, width), nn.Tanh(),
        nn.Linear(width, width),      nn.Tanh(),
        nn.Linear(width, 2),
    )

def make_D(width=64):
    return nn.Sequential(
        nn.Linear(2, width),     nn.LeakyReLU(0.2),
        nn.Linear(width, width), nn.LeakyReLU(0.2),
        nn.Linear(width, 1),     nn.Sigmoid(),
    )

In [ ]:
def gan_train_step(G, D, opt_G, opt_D, real_batch):
    """
    One GAN training step.

    Parameters
    ----------
    G          : Generator network
    D          : Discriminator network
    opt_G      : optimiser for G
    opt_D      : optimiser for D
    real_batch : torch.Tensor, shape (B, 2) — real samples

    Returns
    -------
    l_D : float — discriminator loss value
    l_G : float — generator loss value
    """
    B = real_batch.size(0)

    # ── YOUR CODE HERE ──────────────────────────────────────────────────────
    # ── Step 1: Update D ─────────────────────────────────────────────────
    # a. Sample z ~ N(0,I) of shape (B, LATENT_DIM_GAN)
    # b. Generate fake_batch = G(z).detach()
    # c. Compute d_real = D(real_batch).squeeze()
    #            d_fake = D(fake_batch).squeeze()
    # d. loss_D = discriminator_loss(d_real, d_fake)
    # e. Zero opt_D, backward, step

    # ── Step 2: Update G ─────────────────────────────────────────────────
    # a. Sample new z ~ N(0,I)
    # b. fake_batch = G(z)  (no detach this time)
    # c. d_fake = D(fake_batch).squeeze()
    # d. loss_G = generator_loss(d_fake)
    # e. Zero opt_G, backward, step


    # ────────────────────────────────────────────────────────────────────────

In [ ]:
# ── Run the GAN training ───────────────────────────────────────────────────────
torch.manual_seed(0)
G7 = make_G().to(device)
D7 = make_D().to(device)
opt_G7 = optim.Adam(G7.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D7 = optim.Adam(D7.parameters(), lr=2e-4, betas=(0.5, 0.999))

ring_t  = torch.tensor(ring_data).to(device)
N_STEPS = 3000
B_GAN   = 256
d_hist7, g_hist7 = [], []

for step in range(N_STEPS):
    idx     = torch.randperm(len(ring_t))[:B_GAN]
    real_b  = ring_t[idx]
    l_D, l_G = gan_train_step(G7, D7, opt_G7, opt_D7, real_b)
    d_hist7.append(l_D); g_hist7.append(l_G)

print('Training complete.')

In [ ]:
# ── Test P7 ───────────────────────────────────────────────────────────────────
# Losses must be finite
assert all(np.isfinite(d_hist7)), 'P7 FAIL: D loss contains NaN/Inf'
assert all(np.isfinite(g_hist7)), 'P7 FAIL: G loss contains NaN/Inf'

# Losses should end near log(2) ≈ 0.693 (equilibrium)
d_final = np.mean(d_hist7[-200:])
g_final = np.mean(g_hist7[-200:])
assert 0.3 < d_final < 1.2, f'P7 FAIL: D loss={d_final:.3f} far from equilibrium'
assert 0.3 < g_final < 1.5, f'P7 FAIL: G loss={g_final:.3f} far from equilibrium'

# G should cover multiple modes
G7.eval()
with torch.no_grad():
    z_samp = torch.randn(2000, LATENT_DIM_GAN, device=device)
    gen    = G7(z_samp).cpu().numpy()
G7.train()

modes_covered = sum(
    np.any(np.linalg.norm(gen - c, axis=1) < 0.4)
    for c in ring_centers
)
assert modes_covered >= 5, f'P7 FAIL: only {modes_covered}/8 modes covered'

print(f'P7 PASSED ✓  ({modes_covered}/8 modes covered, D_final={d_final:.3f}, G_final={g_final:.3f})')

# ── Plots ─────────────────────────────────────────────────────────────────────
smooth = lambda arr, w=50: np.convolve(arr, np.ones(w)/w, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(smooth(d_hist7), label='D loss', color='tomato')
axes[0].plot(smooth(g_hist7), label='G loss', color='steelblue')
axes[0].axhline(np.log(2), color='gray', ls='--', lw=1.2, label='log(2) equilibrium')
axes[0].set_title('GAN training losses (smoothed)'); axes[0].legend()

axes[1].scatter(ring_data[:,0], ring_data[:,1], s=5, alpha=0.2, c='steelblue', label='Real')
axes[1].scatter(gen[:,0],       gen[:,1],       s=5, alpha=0.4, c='tomato',    label='Generated')
axes[1].scatter(ring_centers[:,0], ring_centers[:,1], s=60, c='black', zorder=5, label='Modes')
axes[1].set_aspect('equal'); axes[1].set_title('Generated vs. real samples'); axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()

## Problem 8 — Mode Coverage vs. Discriminator Strength

**Mode collapse** occurs when the generator concentrates on only a few modes.
One cause: the discriminator becomes too strong relative to the generator, starving G of gradients.

We can control this imbalance via `d_steps` — the number of discriminator updates per generator update.

**Task:** Train GANs with `d_steps` ∈ {1, 3, 10} on the ring dataset.  
For each, record how many of the 8 modes are covered (a mode is covered if ≥1 generated sample falls within radius 0.4 of its center).  
Plot the generated samples for each case side by side.

In [ ]:
def train_gan_dsteps(ring_t, d_steps=1, n_steps=3000, seed=0):
    """
    Train a GAN with `d_steps` D-updates per G-update.
    Return generated samples (np.ndarray, shape (2000,2)) and list of (d_loss, g_loss).
    """
    torch.manual_seed(seed)
    G_ = make_G().to(device)
    D_ = make_D().to(device)
    oG = optim.Adam(G_.parameters(), lr=2e-4, betas=(0.5, 0.999))
    oD = optim.Adam(D_.parameters(), lr=2e-4, betas=(0.5, 0.999))
    B  = 256

    for step in range(n_steps):
        # ── YOUR CODE HERE ──────────────────────────────────────────────────
        # Run d_steps updates on D, then one update on G.
        # For D updates: sample a fresh real batch and generate new fakes each time.
        # For G update: use gan_train_step (or inline the G-only step).


        # ────────────────────────────────────────────────────────────────────

    G_.eval()
    with torch.no_grad():
        samples = G_(torch.randn(2000, LATENT_DIM_GAN, device=device)).cpu().numpy()
    return samples


# ── YOUR CODE HERE ────────────────────────────────────────────────────────────
# 1. Train for d_steps in [1, 3, 10] and collect generated samples.
# 2. For each, count how many of the 8 modes are covered (threshold 0.4).
# 3. Print results.


# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# ── Test P8 ───────────────────────────────────────────────────────────────────
# d_steps=1 should cover more modes than d_steps=10
try:
    assert modes_1 >= modes_10, \
        f'P8 FAIL: d_steps=1 should cover >= modes than d_steps=10, got {modes_1} vs {modes_10}'
    print(f'P8 PASSED ✓  modes covered: d_steps=1→{modes_1}, d_steps=3→{modes_3}, d_steps=10→{modes_10}')
except NameError:
    print('P8: assign your mode counts to modes_1, modes_3, modes_10 to run this test.')

# ── Plot ─────────────────────────────────────────────────────────────────────
try:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, samples, ds in zip(axes, [samples_1, samples_3, samples_10], [1, 3, 10]):
        ax.scatter(ring_data[:,0], ring_data[:,1], s=4, alpha=0.15, c='steelblue')
        ax.scatter(samples[:,0],   samples[:,1],   s=4, alpha=0.5,  c='tomato')
        ax.scatter(ring_centers[:,0], ring_centers[:,1], s=50, c='black', zorder=5)
        ax.set_aspect('equal')
        ax.set_title(f'd_steps={ds}  ({[modes_1,modes_3,modes_10][[1,3,10].index(ds)]}/8 modes)')
        ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)
    plt.tight_layout(); plt.show()
except NameError:
    print('Plot skipped — assign samples_1, samples_3, samples_10 from train_gan_dsteps.')

print('\n⭐ Advanced: implement one-sided label smoothing and re-run with d_steps=10.')
print('   Replace real labels 1.0 with 0.9 in discriminator_loss and check if more modes are covered.')
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────────────────

**Answer the following (edit this cell):**

1. As `d_steps` increases, what happens to mode coverage and why?
   > *Your answer:*

2. Look at the generated samples for `d_steps=10`. Does the generator collapse to one mode or a few? What does this suggest about the gradient signal G receives?
   > *Your answer:*

3. How does one-sided label smoothing change the training dynamics? (⭐)
   > *Your answer:*

## ⭐ Problem 9 — VAE vs. GAN: Head-to-Head

Both VAE and GAN learn to generate samples from a complex distribution.  
They differ fundamentally in how they measure the gap between $p_G$ and $p_{\text{data}}$:

| Model | Divergence minimised | Sampling quality | Latent structure |
|-------|---------------------|-----------------|------------------|
| VAE   | ELBO (≥ −KL−recon)  | Often blurry    | Smooth, structured |
| GAN   | JSD (via adversary) | Sharp           | Unstructured noise |

**Task:** On the ring dataset, compare the two models side by side:
1. Fit a VAE (use your `VAE2D` with the ring data).
2. Use the trained GAN from Problem 7.
3. For each model, generate 2000 samples and visualise them against the true ring.
4. Count mode coverage for both.

Discuss: which model covers more modes, and why?

In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────
# 1. Normalise ring_data to [0,1] and create a DataLoader.
# 2. Train a VAE2D for 300 epochs (copy the training loop from Problem 4).
# 3. Sample 2000 points from the VAE prior: z ~ N(0,I), decode with vae.decoder.
# 4. Un-normalise the VAE samples back to the ring scale.
# 5. Sample 2000 points from the GAN (already trained in P7).
# 6. Plot: 3-panel figure — Real | VAE samples | GAN samples.
# 7. Report mode coverage for both.


# ─────────────────────────────────────────────────────────────────────────────

**Discussion:**

4. Which model has higher mode coverage on the ring? Why might the VAE spread probability mass differently from the GAN?
   > *Your answer:*

5. The VAE reconstruction loss is pixel-wise MSE/BCE. Explain geometrically why this leads to smoother but less sharp samples compared to the GAN adversarial loss.
   > *Your answer:*

6. When would you prefer a VAE over a GAN in a real application? Give one concrete example for each.
   > *Your answer:*